# Email Spam Detection Using LSTM — Micro-Project (Corrected)

**Colab-ready | Enron-derived Kaggle email text | TensorFlow/Keras**

**Dataset:** [Spam Mails Dataset — Kaggle (venky73)](https://www.kaggle.com/datasets/venky73/spam-mails-dataset). This is **not** the defective CSV previously used. This version uses `spam_ham_dataset.csv` containing full messages.

**Reference:** Saleem et al. (2025), *Spam Email Detection Using Long Short-Term Memory and Gated Recurrent Unit*, *Applied Sciences*, 15(13), 7407, https://doi.org/10.3390/app15137407. Our standalone LSTM is a **micro-project adaptation**, not a reproduction of the paper's experimental results. Optional LSTM–GRU comparison is provided.

**Always run Runtime → Restart session and run all after updating the notebook.** Training results are not pre-computed.

## 0 — Install libraries

In [ ]:
%pip -q install kagglehub tensorflow pandas scikit-learn matplotlib seaborn

## 1 — Imports and reproducibility

In [ ]:
import os, glob, re, json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import tensorflow as tf
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, precision_recall_curve, average_precision_score,
                             accuracy_score, precision_score, recall_score, f1_score)
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, GRU, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

## 2 — Download the **correct** dataset

**Expected:** `spam_ham_dataset.csv` with `label`, `text`, and `label_num` columns; approximately **5,171 original emails** (3,672 ham, 1,499 spam). The notebook checks those values. If a download fails, get the CSV from [Kaggle](https://www.kaggle.com/datasets/venky73/spam-mails-dataset) and upload it manually using the Colab Files sidebar, then re-run this cell.

The previous Enron CSV had thousands of spam labels but almost no usable spam message text after cleaning. **Do not use that earlier file for this notebook.**

In [ ]:
CSV_NAME = 'spam_ham_dataset.csv'
local = Path('/content') / CSV_NAME
if local.exists():
    csv_path = str(local)
    print('Using uploaded CSV:', csv_path)
else:
    try:
        folder = kagglehub.dataset_download('venky73/spam-mails-dataset')
        matches = glob.glob(os.path.join(folder, '**', CSV_NAME), recursive=True)
        if not matches:
            raise FileNotFoundError(f'{CSV_NAME} not found in {folder}')
        csv_path = matches[0]
        print('Downloaded:', csv_path)
    except Exception as exc:
        raise RuntimeError(
            'Kaggle download failed. Download spam_ham_dataset.csv from '
            'https://www.kaggle.com/datasets/venky73/spam-mails-dataset, '
            'upload it into /content in Colab, and rerun this cell.'
        ) from exc

df = pd.read_csv(csv_path, encoding='utf-8', encoding_errors='replace', low_memory=False)
print('Raw dimensions:', df.shape)
print('Columns:', df.columns.tolist())
assert {'label','text'}.issubset(df.columns), 'Wrong dataset: expected columns label and text.'
print('Raw labels:')
print(df['label'].astype(str).str.lower().str.strip().value_counts(dropna=False))
display(df[['label','text']].head(3))

## 3 — Clean email text and validate BOTH classes

Unlike the old notebook, this dataset has actual message text for spam and ham. We remove exact duplicate text **before splitting** so duplicated emails cannot leak into the test set. An explicit check stops the notebook if either class collapses.

In [ ]:
def clean_text(s):
    s = str(s).lower()
    s = re.sub(r'<[^>]+>', ' ', s)               # Remove HTML tags
    s = re.sub(r'https?://\S+|www\.\S+', ' urltoken ', s)
    s = re.sub(r'[\w.%-]+@[\w.-]+\.[a-z]{2,}', ' emailtoken ', s)
    s = re.sub(r'[^a-z0-9 ]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

label_strings = df['label'].astype('string').str.strip().str.lower()
assert label_strings.isin(['ham', 'spam']).all(), 'Unrecognized labels in CSV.'
data = pd.DataFrame({
    'text': df['text'].fillna('').astype(str),
    'label': label_strings.map({'ham': 0, 'spam': 1}).astype(int)
})
data['clean'] = data['text'].map(clean_text)
print('Before preprocessing:', data['label'].value_counts().sort_index().to_dict())
data = data.loc[data['clean'].str.len() > 0].copy()

# Conflicting duplicate text must never appear in both train and test.
ambiguous = data.groupby('clean')['label'].nunique()
conflicts = set(ambiguous.index[ambiguous > 1])
print('Identical cleaned texts with conflicting labels:', len(conflicts))
if conflicts:
    data = data.loc[~data['clean'].isin(conflicts)].copy()

before = len(data)
data = data.drop_duplicates(subset=['clean'], keep='first').reset_index(drop=True)
counts = data['label'].value_counts().sort_index()
print('Exact duplicates removed:', before - len(data))
print('After preprocessing (0=ham, 1=spam):', counts.to_dict())
assert counts.get(0, 0) >= 1000 and counts.get(1, 0) >= 1000, (
    'STOP: class collapse. Make sure the correct spam_ham_dataset.csv is loaded.'
)
assert data['clean'].nunique() == len(data), 'Duplicate texts remain.'
print('Class validation passed. Dataset is ready.')

## 4 — Exploratory data analysis

In [ ]:
data['word_count'] = data['clean'].str.split().str.len()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
counts = data['label'].map({0:'Ham',1:'Spam'}).value_counts().reindex(['Ham','Spam'])
sns.barplot(x=counts.index, y=counts.values, ax=ax[0])
ax[0].set(title='Class distribution after cleaning', xlabel='Email class', ylabel='Email count')
sns.histplot(data=data.assign(Class=data['label'].map({0:'Ham',1:'Spam'})),
             x='word_count', hue='Class', bins=45, ax=ax[1])
ax[1].set(xlim=(0,400), title='Email length distribution', xlabel='Words')
plt.tight_layout()
plt.show()
print('Email word-count summary:')
display(data.groupby('label')['word_count'].describe().round(1))

## 5 — Stratified training / validation / test split

**64% train / 16% validation / 20% test**, approximately. The tokenizer fits **only training text** (avoiding data leakage). We keep the test partition untouched until final evaluation.

In [ ]:
X = data['clean'].to_numpy()
y = data['label'].to_numpy()
X_dev, X_test_text, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train_text, X_val_text, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.20, stratify=y_dev, random_state=SEED)
for name, values in [('Train', y_train),('Validation',y_val),('Test',y_test)]:
    c = pd.Series(values).value_counts()
    print(f'{name}: {len(values)} emails; ham={c.get(0,0)}; spam={c.get(1,0)}')
    assert c.get(0,0) >= 100 and c.get(1,0) >= 100, 'Split has too few examples of one class'

VOCAB = 10000
MAXLEN = 250
tokenizer = Tokenizer(num_words=VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)
def encode(texts):
    return pad_sequences(tokenizer.texts_to_sequences(list(texts)),
                         maxlen=MAXLEN, padding='post', truncating='post')
X_train = encode(X_train_text)
X_val = encode(X_val_text)
X_test = encode(X_test_text)
print('Encoded shapes:', X_train.shape, X_val.shape, X_test.shape)

## 6 — Train LSTM

Embedding → LSTM → Dense → Dropout → Sigmoid. EarlyStopping restores the best validation-loss weights; do not infer that the model is good from accuracy alone.

In [ ]:
def build_model(hybrid=False):
    net = Sequential(name='LSTM_GRU' if hybrid else 'LSTM')
    net.add(Input(shape=(MAXLEN,)))
    net.add(Embedding(input_dim=VOCAB, output_dim=64, mask_zero=True))
    net.add(LSTM(64, return_sequences=hybrid))
    if hybrid:
        net.add(GRU(32))
    net.add(Dense(64, activation='relu'))
    net.add(Dropout(0.5))
    net.add(Dense(1, activation='sigmoid'))
    net.compile(optimizer='adam', loss='binary_crossentropy', metrics=[
        'accuracy', tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')])
    return net

tf.keras.backend.clear_session()
lstm = build_model()
lstm.summary()
history = lstm.fit(X_train, y_train,
                   validation_data=(X_val,y_val),
                   epochs=12, batch_size=64,
                   callbacks=[EarlyStopping(monitor='val_loss', patience=3,
                                            restore_best_weights=True)], verbose=1)

## 7 — Training graphs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key in zip(axes, ['loss','accuracy']):
    ax.plot(history.history[key], label='Train')
    ax.plot(history.history['val_' + key], label='Validation')
    ax.set(xlabel='Epoch', ylabel=key.title(), title=key.title()+' by epoch')
    ax.legend()
plt.tight_layout()
plt.show()

## 8 — Test performance (real spam detection)

A model predicting HAM for every message can appear accurate on a mostly HAM dataset. We inspect spam **precision, recall, F1**, confusion matrix, ROC-AUC, and average precision. The model must predict both classes.

In [ ]:
def evaluate(model, model_name):
    probs = model.predict(X_test, verbose=0).ravel()
    preds = (probs >= 0.5).astype(int)
    print(f'\n{model_name} classification report:')
    print(classification_report(y_test, preds, target_names=['Ham','Spam'], zero_division=0))
    rocauc = roc_auc_score(y_test, probs)
    ap = average_precision_score(y_test, probs)
    print(f'ROC-AUC: {rocauc:.4f} | Average precision: {ap:.4f}')
    cm = confusion_matrix(y_test, preds, labels=[0,1])
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Ham','Spam'], yticklabels=['Ham','Spam'], ax=ax[0])
    ax[0].set(title=model_name+' confusion matrix', xlabel='Predicted', ylabel='Actual')
    fpr, tpr, _ = roc_curve(y_test, probs)
    ax[1].plot(fpr, tpr, label=f'ROC-AUC = {rocauc:.3f}')
    ax[1].plot([0,1],[0,1], '--', c='gray')
    ax[1].set(title='ROC curve', xlabel='False positive rate', ylabel='True positive rate')
    ax[1].legend()
    prec, rec, _ = precision_recall_curve(y_test, probs)
    ax[2].plot(rec, prec, label=f'Avg. precision = {ap:.3f}')
    ax[2].set(title='Precision–Recall curve', xlabel='Spam recall', ylabel='Spam precision',
              xlim=(0,1), ylim=(0,1.05))
    ax[2].legend()
    plt.tight_layout()
    plt.show()
    stats = {'Model': model_name,
             'Accuracy': accuracy_score(y_test, preds),
             'Spam Precision': precision_score(y_test, preds, zero_division=0),
             'Spam Recall': recall_score(y_test, preds, zero_division=0),
             'Spam F1': f1_score(y_test, preds, zero_division=0),
             'ROC-AUC': rocauc, 'Average Precision': ap}
    if stats['Spam Recall'] < 0.5:
        warnings.warn('LOW SPAM RECALL: review model quality before reporting success.')
    return stats

results = [evaluate(lstm, 'LSTM')]
display(pd.DataFrame(results).round(4))

## 9 — Optional paper-inspired LSTM–GRU comparison

Set `RUN_HYBRID=True` to train a second model. This is an illustrative adaptation of the paper, **not an exact reproduction**. Default `False` keeps runs short.

In [ ]:
RUN_HYBRID = False
results = [r for r in results if r['Model'] != 'LSTM-GRU']
if RUN_HYBRID:
    tf.keras.utils.set_random_seed(SEED + 1)
    hybrid_model = build_model(hybrid=True)
    hybrid_model.fit(X_train, y_train, validation_data=(X_val,y_val),
                     epochs=12, batch_size=64,
                     callbacks=[EarlyStopping(monitor='val_loss', patience=3,
                                              restore_best_weights=True)], verbose=1)
    results.append(evaluate(hybrid_model, 'LSTM-GRU'))
display(pd.DataFrame(results).round(4))

## 10 — Test custom email messages and save the trained model

Predictions are illustrative and should not be treated as guarantees about email safety.

In [ ]:
def predict_email(message, model=None):
    if model is None:
        model = lstm
    score = float(model.predict(encode([clean_text(message)]), verbose=0)[0,0])
    return {'prediction': 'SPAM' if score >= 0.5 else 'HAM',
            'spam_probability': round(score, 4)}

samples = [
    'Congratulations! You won a free cash prize. Claim your money now!',
    'Hi, the project meeting is at 2 PM tomorrow. Please bring the notes.',
    'URGENT: verify your account to claim a bonus reward today!'
]
for sample in samples:
    print(sample, '->', predict_email(sample))

lstm.save('email_spam_lstm.keras')
with open('tokenizer.json', 'w', encoding='utf-8') as f:
    f.write(tokenizer.to_json())
with open('config.json', 'w', encoding='utf-8') as f:
    json.dump({'vocab':VOCAB,'maxlen':MAXLEN,'seed':SEED}, f)
pd.DataFrame(results).to_csv('test_metrics.csv', index=False)
print('Saved: email_spam_lstm.keras, tokenizer.json, config.json, test_metrics.csv')

## 11 — Final checks before GitHub upload

- The CSV should have ~5,171 emails before cleaning, including ~1,499 spam.
- Both classes must have **hundreds of test examples**.
- **Never claim 99% accuracy proves successful spam detection** without spam recall and F1.
- The paper uses a different full Enron collection and hybrid LSTM–GRU. Cite the paper as inspiration rather than claiming identical results.
- Don't commit model files, Kaggle dataset copies, or test emails containing personal information without reviewing permissions/privacy.
- If you change code, select **Runtime → Restart session and run all**. The notebook was statically validated; Colab runtime performance depends on downloads and available compute.